# BEE 4750 Homework 5: Mixed Integer and Stochastic Programming

**Name**:

**ID**:

> **Due Date**
>
> Thursday, 12/04/24, 9:00pm

## Overview

### Instructions

-   In Problem 1, you will use mixed integer programming to solve a
    waste load allocation problem.
-   In Problem 2, you will formulate a stochastic optimization problem.

### Load Environment

The following code loads the environment and makes sure all needed
packages are installed. This should be at the start of most Julia
scripts.

In [1]:
import Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

  Activating project at `c:\Users\holly\4750 HW\hw5-ellasand`


In [3]:
using JuMP
using HiGHS
using DataFrames
using GraphRecipes
using Plots
using Measures
using MarkdownTables

## Problems (Total: 30 Points)

### Problem 1 (24 points)

Three cities are developing a coordinated municipal solid waste (MSW)
disposal plan. Three disposal alternatives are being considered: a
landfill (LF), a materials recycling facility (MRF), and a
waste-to-energy facility (WTE). The capacities of these facilities and
the fees for operation and disposal are provided below.

-   **LF**: Capacity 200 Mg, fixed cost \$2000/day, tipping cost
    \$50/Mg;
-   **MRF**: Capacity 350 Mg, fixed cost \$1500/day, tipping cost
    \$7/Mg, recycling cost \$40/Mg recycled;
-   **WTE**: Capacity 210 Mg, fixed cost \$2500/day, tipping cost
    \$60/Mg;

The MRF recycling rate is 40%, and the ash fraction of non-recycled
waste is 16% and of recycled waste is 14%. Transportation costs are
\$1.5/Mg-km, and the relative distances between the cities and
facilities are provided in the table below.

| **City/Facility** | **Landfill (km)** | **MRF (km)** | **WTE (km)** |
|:-----------------:|:-----------------:|:------------:|:------------:|
|         1         |         5         |      30      |      15      |
|         2         |        15         |      25      |      10      |
|         3         |        13         |      45      |      20      |
|        LF         |        \-         |      32      |      18      |
|        MRF        |        32         |      \-      |      15      |
|        WTE        |        18         |      15      |      \-      |

The fixed costs associated with the disposal options are incurred only
if the particular disposal option is implemented. The three cities
produce 100, 90, and 120 Mg/day of solid waste, respectively, with the
composition provided in the table below.

| **Component** | **% of total mass** | **Combustion ash** (%) | **MRF Recycling rate** (%) |
|:---------------------:|:--------------:|:---------------:|:---------------:|
| Food Wastes | 15 | 8 | 0 |
| Paper & Cardboard | 40 | 7 | 55 |
| Plastics | 5 | 5 | 15 |
| Textiles | 3 | 10 | 10 |
| Rubber, Leather | 2 | 15 | 0 |
| Wood | 5 | 2 | 30 |
| Yard Wastes | 18 | 2 | 40 |
| Glass | 4 | 100 | 60 |
| Ferrous | 2 | 100 | 75 |
| Aluminum | 2 | 100 | 80 |
| Other Metal | 1 | 100 | 50 |
| Miscellaneous | 3 | 70 | 0 |

The information in the above table will help you determine the overall
recycling and ash fractions. Note that the recycling residuals, which
may be sent to either landfill or the WTE, have different ash content
than the ash content of the original MSW. You will need to determine
these fractions to construct your mass balance constraints.

**Reminder**: Use `round(x; digits=n)` to report values to the
appropriate precision!

#### Problem 1.1

Based on the information above, calculate the overall recycling and ash
fractions for the waste produced by each city.

Using excel, I found that the overall ash fraction for initial waste to be 16.4%, and the overall recycling rate to be 38%. Considering the ash content of the recycled waste, we multiplied each recycled material by the corresponding fraction of ash, and got a residual recycled waste fraction of 13.9%, which is lower than the non-recycled waste. 

#### Problem 1.2

What are the decision variables for your optimization problem? Provide
notation and variable meaning.

Our decision variables are:

Wij, waste transported from city i to facility j (9 of these)

Rkj, residual waste going from disposal facility k to j (3 of these, recycle to WTE, recycling to landfill, and WTE to landfill)

Yj, binary variable to determine if facility j is operating


#### Problem 1.3

Formulate the objective function. Make sure to include any needed
derivations or justifications for your equation(s).

We want to minimize cost, so we need to combine transportation and disposal costs. if landfill=1, MRF=2, and WTE=3, transportation costs:

1.5[5W11+30W12+15W13+15W21+25W22+10W23+13W31+
45W32+20W33+32R21+18R31+15R23]

As for disposal costs:

LF: 2000Y1 + 50(W11+W21+W31+R21+R31)

MRF: 1500Y2 + 7(W12+W22+W32)+0.38(40)(W12+W22+W32)

WTE: 2500Y3 + 60(W13+W23+W33+R23)

Combining these, we get:

2000Y1+1500Y2+2500Y3+57.5W11+67.2W12+82.5W13+72.5W21+59.7W22

+75W23+69.5W31+89.7W32+90W33+98R31+77R31+82.5R23


#### Problem 1.4

Derive all relevant constraints. Make sure to include any needed
justifications or derivations.

There are mass-balance constraints, capacity constraints, non-negativity constraints, binary constraints, and demand constraints, where all waste from each city must be treated. 
Mass-balance:

For the ash mass balance, we must consider that waste sent directly to WTE has an ash fraction of 0.164, but waste sent from recycling to WTE (R23) has an ash content of 0.14. Therefore:
0.164(W13+W23+W33)+0.14(R23) = R31

As for recycling mass-balance: R21+R23 = (1-0.38)(W12+W22+W32)

Capacity:

W11+W21+W31+R21+R31 <= 200 (LF)

W12+W22+W32 <= 350 (MRF)

W13+W23+W33+R23 <= 210 (WTE)

Demand:

W11+W12+W13 = 100 (city 1)

W21+W22+W23 = 90 (city 2)

W31+W32+W33 = 120 (city 3)

Binary: I will set Y1 = 1, since the landfill always has to operate, and I will use big M formulation, as follows:

W12+W22+W32 <= M2*Y2; M2 = 350

W13+W23+W33+R23 <= M3*Y3, M3 = 210

For non-negativity, I will define the variables as being >=0.

#### Problem 1.5

Find the optimal solution (using `JuMP` to solve the problem). Report
the optimal objective value.

In [3]:
using JuMP
using HiGHS
using DataFrames
using GraphRecipes
using Plots
using Measures
using MarkdownTables

M2 = 350
M3 = 210

waste_model = Model(HiGHS.Optimizer) # initialize model object
@variable(waste_model, W[1:9] >= 0) # non-negativity constraints
@variable(waste_model, R[1:3] >= 0)

# Y: binary commitment variables (1 = facility open, 0 = closed)
@variable(waste_model, Y[1:3], Bin)
@constraint(waste_model, commit2, W[2] + W[5] + W[8] <= M2 * Y[2])
@constraint(waste_model, commit3, W[3] + W[6] + W[9] + R[3] <= M3 * Y[3])
# landfill always has to be operating
@constraint(waste_model, commit1, Y[1] == 1)
# cost function
@objective(waste_model, Min, 2000*Y[1]+1500*Y[2]+2500*Y[3]+57.5*W[1]+67.2*W[2]+82.5*W[3]+72.5*W[4]
+59.7*W[5]+75*W[6]+69.5*W[7]+89.7*W[8]+90*W[9]+98*R[1]+77*R[2]+82.5*R[3])
# capacity constraints
@constraint(waste_model, cap1, (W[1]+W[4]+W[7]+sum(R[1:2])<=200))
@constraint(waste_model, cap2, (W[2]+W[5]+W[8]<=350))
@constraint(waste_model, cap3, (W[3]+W[6]+W[9]+R[3]<=210))
# waste demand constraints
@constraint(waste_model, dem1, (sum(W[1:3]) == 100))
@constraint(waste_model, dem2, (sum(W[4:6]) == 90))
@constraint(waste_model, dem3, (sum(W[7:9]) == 120))
# mass balance constraints
@constraint(waste_model, mass_mrf, (0.62*(W[2]+W[5]+W[8]) == R[1]+R[3]))
@constraint(waste_model, mass_wte, (0.164*(W[3]+W[6]+W[9])+0.14*(R[3]) == R[2]))

optimize!(waste_model)
display(value.(Y))
display(value.(W))
display(value.(R))
total_cost = objective_value.(waste_model)
display(total_cost)


3-element Vector{Float64}:
  1.0
 -0.0
  1.0

9-element Vector{Float64}:
 100.0
   0.0
   0.0
  -0.0
  -0.0
  90.0
  78.42105263157897
   0.0
  41.57894736842095

3-element Vector{Float64}:
  0.0
 21.578947368421034
  0.0

27853.947368421042

Running HiGHS 1.12.0 (git hash: 755a8e027): Copyright (c) 2025 HiGHS under MIT licence terms
MIP has 11 rows; 15 cols; 41 nonzeros; 3 integer variables (3 binary)
Coefficient ranges:
  Matrix  [1e-01, 4e+02]
  Cost    [6e+01, 2e+03]
  Bound   [1e+00, 1e+00]
  RHS     [1e+00, 4e+02]
Presolving model
9 rows, 14 cols, 37 nonzeros  0s
8 rows, 13 cols, 35 nonzeros  0s
Presolve reductions: rows 8(-3); columns 13(-2); nonzeros 35(-6) 

Solving MIP model with:
   8 rows
   13 cols (2 binary, 0 integer, 0 implied int., 11 continuous, 0 domain fixed)
   35 nonzeros

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y => HiGHS solution;
     Z => ZI Round; l => Trivial lower; p => Trivial point; u => Trivial upper; z => Trivial zero

        Nodes      |    B&B Tree     |            Objective

Without using the MRF facility, the minimum objective is $27,793 per day. City one moves everything straight to the landfill, city 2 only uses WTE, and city 3 splits between both (78 to landfill, 42 to WTE). 

#### Problem 1.6

Draw a diagram showing the flows of waste between the cities and the
facilities. Which facilities (if any) will not be used? Does this
solution make sense?

We do not use the MRF facility, which is interesting because the operational costs are pretty low. However, the transportation costs are high seeing as this facility is pretty far from the 3 cities. This may help explain why it is best to not use the recycling facility.

### Problem 2 (6 points)

Consider a two-period economic dispatch problem, based on the
multi-period example from Lecture 14 (on 10/29). The generator data,
including ramping constraints for each generator, is provided in
\`data/generators.csv.’ In period 1, the demand is
$d_1 = 1100 \text{MW}$. In period 2, the demand is projected to be
$d_2 = 1200 \text{MW}$, but there is a 25% probability that it is \$1500
. In the first period, the solar capacity factor is $0.9$ and the wind
capacity factor is $0.45$, but in the second period, there is some
uncertainty: the forecasted solar and wind capacity factors are $0.95$
and $0.4$, respectively, but there is a 30% probability that they are
$0.75$ and $0.5$. Your goal is to identify how to dispatch your
generators to minimize the cost of meeting demand.

#### Problem 2.1

Draw a scenario tree for this problem.

#### Problem 2.2

Formulate a stochastic linear program for this problem based on your
scenario tree from Problem 2.1 and the data in `data/generators.csv`.

## References

List any external references consulted, including classmates.